In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
import os

In [2]:
def get_spark():    
    spark = SparkSession.builder \
        .master("spark://spark-master:7077") \
        .appName('Trino Delta Lake') \
        .config("spark.jars.packages",
                "org.apache.hadoop:hadoop-aws:3.2.2,"
                "io.delta:delta-spark_2.12:3.2.0,"
                "io.delta:delta-storage:3.2.0,"
                "com.amazonaws:aws-java-sdk-bundle:1.12.180") \
        .config("spark.executor.memory", "4g") \
        .config("spark.executor.cores", "4") \
        .config("spark.driver.memory", "2g") \
        .config("spark.driver.cores", "2") \
    	.config("spark.dynamicAllocation.enabled", "true") \
        .config("spark.dynamicAllocation.minExecutors", "1") \
        .config("spark.dynamicAllocation.maxExecutors", "4") \
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
        .config("spark.hadoop.fs.s3a.access.key", "minio") \
        .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
        .config("spark.hadoop.fs.s3a.path.style.access", "true") \
        .config("spark.hadoop.com.amazonaws.services.s3.enableV4", "true") \
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
        .config("spark.hadoop.fs.AbstractFileSystem.s3a.impl", "org.apache.hadoop.fs.s3a.S3A") \
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
        .getOrCreate()
    
    spark.sparkContext.setLogLevel("DEBUG")
    
    return spark
    
spark = get_spark()

In [3]:
spark.catalog.clearCache()

In [4]:
schema = StructType([
    StructField("ID_PESSOA", IntegerType(), False),
    StructField("NOME", StringType(), False),
    StructField("CPF", StringType(), False),
    StructField("EMAIL", StringType(), True)
])

In [5]:
data = [
    (1, "João Silva", "12345678900", "joao@email.com"),
    (2, "Maria Souza", "98765432100", "maria@email.com"),
    (3, "Carlos Santos", "45678912300", "carlos@email.com")
]

In [6]:
df = spark.createDataFrame(data,schema=schema)

In [7]:
df.show(10)

+---------+-------------+-----------+----------------+
|ID_PESSOA|         NOME|        CPF|           EMAIL|
+---------+-------------+-----------+----------------+
|        1|   João Silva|12345678900|  joao@email.com|
|        2|  Maria Souza|98765432100| maria@email.com|
|        3|Carlos Santos|45678912300|carlos@email.com|
+---------+-------------+-----------+----------------+



In [8]:
df.write.format("delta").save("s3a://bronze/delta/pessoa", mode="overwrite")

In [9]:
spark.sql(f"CREATE TABLE IF NOT EXISTS pessoas USING DELTA LOCATION 's3a://bronze/delta/pessoa' ")

DataFrame[]

In [5]:
df_lido = spark.read.format("delta").load("s3a://bronze/delta/pessoa")
df_lido.show()

+---------+--------------------+--------------+--------------------+
|ID_PESSOA|                NOME|           CPF|               EMAIL|
+---------+--------------------+--------------+--------------------+
|        1|          João Silva|   12345678900|      joao@email.com|
|        2|         Maria Souza|   98765432100|     maria@email.com|
|        3|       Carlos Santos|   45678912300|    carlos@email.com|
|        4|Luiz Antonio Rous...|356.787.900-64|luiz@roussenq.com.br|
|        5|     Renata Roussenq|355.711.917-35|renata@roussenq.c...|
+---------+--------------------+--------------+--------------------+



In [8]:
spark.stop()